# Step 6: 모델 배포 & 데모

Fine-tuned 모델을 SageMaker Endpoint로 배포하고 Gradio 데모를 실행합니다.

In [ ]:
import json
import os
import sagemaker
from sagemaker.pytorch import PyTorchModel
from datetime import datetime
from pathlib import Path

# ============================================
# 프로젝트 경로 자동 설정
# ============================================
home_dir = Path.home()
PROJECT_ROOT = home_dir / 'deepfake-detection-sagemaker'
notebook_dir = PROJECT_ROOT / '6_demo'
os.chdir(notebook_dir)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Current Dir: {os.getcwd()}")

# 설정 로드
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

sagemaker_session = sagemaker.Session()
role = config['role']

# 고유한 Endpoint 이름 생성 (충돌 방지)
ENDPOINT_NAME = f"deepfake-detector-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
print(f"Endpoint Name: {ENDPOINT_NAME}")
print(f"Config loaded: {config_path}")

In [ ]:
# SageMaker Endpoint 배포
pytorch_model = PyTorchModel(
    model_data=config['model_data'],
    role=role,
    entry_point='inference.py',
    source_dir='.',
    framework_version='2.0.0',
    py_version='py310'
)

print("Endpoint 배포 중... (약 5-10분 소요)")
predictor = pytorch_model.deploy(
    initial_instance_count=1,
    instance_type='ml.g4dn.xlarge',
    endpoint_name=ENDPOINT_NAME
)

print(f"✅ Endpoint 배포 완료: {predictor.endpoint_name}")

# config에 endpoint 이름 저장
config['endpoint_name'] = ENDPOINT_NAME
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"Endpoint 이름이 config.json에 저장되었습니다.")

In [ ]:
# Gradio 데모 실행 (노트북 내에서)
import gradio as gr
import boto3
import base64
from io import BytesIO
from PIL import Image

runtime = boto3.client('sagemaker-runtime')

def detect_deepfake(image):
    """딥페이크 탐지 함수"""
    if image is None:
        return "이미지를 업로드해주세요."
    
    # 이미지를 base64로 인코딩
    buffered = BytesIO()
    image.save(buffered, format="JPEG")
    img_bytes = buffered.getvalue()
    
    # Endpoint 호출
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='application/x-image',
        Body=img_bytes
    )
    
    result = json.loads(response['Body'].read().decode())
    
    # 결과 포맷팅
    prediction = result.get('prediction', 'Unknown')
    confidence = result.get('confidence', 0)
    
    if prediction == 'FAKE':
        return f"🚨 FAKE 탐지!\n확신도: {confidence:.1%}"
    else:
        return f"✅ REAL\n확신도: {confidence:.1%}"

# Gradio 인터페이스
demo = gr.Interface(
    fn=detect_deepfake,
    inputs=gr.Image(type="pil", label="이미지 업로드"),
    outputs=gr.Textbox(label="탐지 결과"),
    title="🎭 딥페이크 탐지 데모",
    description="이미지를 업로드하면 딥페이크 여부를 판별합니다.\n(KoDF Fine-tuned 모델 사용)"
)

# 노트북에서 실행
demo.launch(share=True)

In [ ]:
# ⚠️ 실습 완료 후 반드시 실행하세요! (비용 절감)
# 아래 주석을 해제하고 실행하면 Endpoint가 삭제됩니다.

# predictor.delete_endpoint()
# print(f"✅ Endpoint '{ENDPOINT_NAME}' 삭제 완료!")
# print("더 이상 비용이 발생하지 않습니다.")